# Song Lyrics Topic Classification (1950–2019)

**NLP Project — Music Dataset (Kaggle)**

This notebook implements a full NLP pipeline to classify song lyrics into eight topic categories:
`sadness`, `violence`, `romantic`, `music`, `night/time`, `feelings`, `obscene`, `world/life`.

**Models implemented:**
- Traditional ML: Random Forest, Naïve Bayes
- Deep Learning: RNN (SimpleRNN), LSTM

**Dataset:** [Music Dataset 1950–2019 on Kaggle](https://www.kaggle.com/datasets/saurabhshahane/music-dataset-1950-to-2019)

## Rubric Coverage (Quick Checklist)

- **Scope & objective:** Classify song lyrics (Kaggle Music Dataset 1950–2019) into eight topic labels.
- **Data description:** Loads `music_dataset.csv` with `lyrics/topic` columns; reports usable rows and class counts.
- **Preprocessing & EDA:** Cleaning, stopword removal, optional stemming, label encoding; visuals for class distribution, length violin plot, wordclouds, and TF-IDF heatmap.
- **Baselines:** Majority-class dummy, Multinomial Naïve Bayes (GridSearch on TF-IDF hyperparams), and Random Forest (balanced class weights).
- **Deep learning:** Simple RNN and LSTM with embedding + padding, tuned on validation and evaluated on test.
- **Metrics & visuals:** Accuracy, macro F1, ROC-AUC (OvR), confusion matrices, and a final results table.
- **Reproducibility:** Fixed `RANDOM_STATE`, documented pip requirements (NumPy/Pandas/SciPy/SKLearn/Matplotlib/Seaborn/WordCloud/TensorFlow).
- **How to run:** Place `music_dataset.csv` beside the notebook, run cells top-to-bottom; GPU is optional (small models).


## 1. Imports & Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import re
import warnings
import os

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# ── Deep learning ─────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, LSTM, Dense, Dropout, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# ── Download NLTK resources ───────────────────────────────────────────────────
for resource in ['stopwords', 'punkt', 'punkt_tab']:
    nltk.download(resource, quiet=True)

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")

## 2. Dataset Loading & Description

The **Music Dataset 1950–2019** (Saurabhshahane, Kaggle) contains over 28,000 songs with lyrics and
pre-computed audio/text features. The target column `topic` has eight classes.

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
# Download CSV from Kaggle or place 'music_dataset.csv' in the working directory.
# Fallback: if running in Kaggle notebooks, use the built-in path.

def load_music_dataset(paths=None):
    """Attempt to load the Music Dataset from several candidate paths."""
    if paths is None:
        paths = [
            'music_dataset.csv',
            '/kaggle/input/music-dataset-1950-to-2019/music_dataset_cleaned.csv',
            '/kaggle/input/music-dataset-1950-to-2019/tcc_ceds_music.csv',
        ]
    for p in paths:
        if os.path.exists(p):
            print(f"Loading dataset from: {p}")
            return pd.read_csv(p)
    raise FileNotFoundError(
        "Dataset not found. Please download 'tcc_ceds_music.csv' from "
        "https://www.kaggle.com/datasets/saurabhshahane/music-dataset-1950-to-2019 "
        "and place it in the working directory as 'music_dataset.csv'."
    )

df_raw = load_music_dataset()
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
print("Columns:", df_raw.columns.tolist())
print("\nDtypes:")
print(df_raw.dtypes)
print("\nMissing values per column:")
print(df_raw.isnull().sum())

In [ ]:
# ── Keep only relevant columns ────────────────────────────────────────────────
df = df_raw[['lyrics', 'topic']].dropna().copy()
df.columns = ['text', 'label']

# Standardise label strings
df['label'] = df['label'].str.strip().str.lower()

print(f"Usable rows after dropping NaN: {len(df)}")
print("\nClass distribution:")
print(df['label'].value_counts())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Visualisation 1: Class distribution ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
counts = df['label'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=sns.color_palette('Set2', len(counts)))
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title('Class Distribution of Song Topics', fontsize=14, fontweight='bold')
ax.set_xlabel('Topic', fontsize=11)
ax.set_ylabel('Number of Songs', fontsize=11)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('fig_class_distribution.png', dpi=150)
plt.show()
print("Figure saved: fig_class_distribution.png")

In [ ]:
# ── Visualisation 2: Lyrics length distribution per topic ─────────────────────
df['text_len'] = df['text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 5))
topic_order = df['label'].value_counts().index.tolist()
data_for_violin = [df.loc[df['label'] == t, 'text_len'].values for t in topic_order]
parts = ax.violinplot(data_for_violin, showmedians=True)
ax.set_xticks(range(1, len(topic_order) + 1))
ax.set_xticklabels(topic_order, rotation=30, ha='right')
ax.set_title('Lyrics Length Distribution by Topic', fontsize=14, fontweight='bold')
ax.set_xlabel('Topic', fontsize=11)
ax.set_ylabel('Word Count', fontsize=11)
plt.tight_layout()
plt.savefig('fig_length_distribution.png', dpi=150)
plt.show()
print("Figure saved: fig_length_distribution.png")

In [ ]:
# ── Visualisation 3: Word clouds for two contrasting topics ───────────────────
def make_wordcloud(texts, title, ax, colormap='Blues'):
    """Generate and render a word cloud on the given axes."""
    combined = ' '.join(texts.dropna().values)
    wc = WordCloud(
        width=600, height=300,
        background_color='white',
        colormap=colormap,
        max_words=100,
        stopwords=set(stopwords.words('english'))
    ).generate(combined)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
make_wordcloud(df.loc[df['label'] == 'romantic', 'text'], 'Romantic', axes[0], 'RdPu')
make_wordcloud(df.loc[df['label'] == 'sadness', 'text'],  'Sadness',  axes[1], 'Blues')
plt.suptitle('Word Clouds by Topic', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: fig_wordclouds.png")

In [ ]:
# ── Visualisation 4: TF-IDF top-10 terms per topic (heatmap) ─────────────────
def tfidf_top_terms(corpus, labels, n_terms=10):
    """Return a DataFrame with the mean TF-IDF score of the top-n terms per label."""
    vec = TfidfVectorizer(max_features=5000, stop_words='english', min_df=5)
    X = vec.fit_transform(corpus)
    terms = vec.get_feature_names_out()
    result = {}
    for lbl in sorted(labels.unique()):
        mask = (labels == lbl).values
        mean_tfidf = X[mask].mean(axis=0).A1
        top_idx = np.argsort(mean_tfidf)[::-1][:n_terms]
        result[lbl] = {terms[i]: mean_tfidf[i] for i in top_idx}
    return pd.DataFrame(result).fillna(0)

tfidf_df = tfidf_top_terms(df['text'], df['label'], n_terms=10)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(tfidf_df, cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Mean TF-IDF Score — Top-10 Terms per Topic', fontsize=13, fontweight='bold')
ax.set_xlabel('Topic', fontsize=11)
ax.set_ylabel('Term', fontsize=11)
plt.tight_layout()
plt.savefig('fig_tfidf_heatmap.png', dpi=150)
plt.show()
print("Figure saved: fig_tfidf_heatmap.png")

## 4. Preprocessing Pipeline

In [ ]:
# ── Text cleaning ─────────────────────────────────────────────────────────────
STOP_WORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()


def clean_text(text, stem=False):
    """
    Lowercase, remove non-alphabetic characters, tokenise,
    remove stopwords, and optionally apply stemming.
    """
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    if stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)


# Apply cleaning (stem=False keeps words readable for word clouds / TF-IDF)
df['clean_text'] = df['text'].apply(clean_text)

# Show example
idx = 0
print("Original:\n", df['text'].iloc[idx][:300])
print("\nCleaned:\n", df['clean_text'].iloc[idx][:300])

In [ ]:
# ── Label encoding ────────────────────────────────────────────────────────────
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
NUM_CLASSES = len(le.classes_)
print(f"Number of classes : {NUM_CLASSES}")
print("Mapping           :", dict(enumerate(le.classes_)))

In [ ]:
# ── Train / validation / test split (70 / 15 / 15) ───────────────────────────
X = df['clean_text'].values
y = df['label_enc'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train : {len(X_train):,}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val   : {len(X_val):,}  ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test  : {len(X_test):,}  ({len(X_test)/len(X)*100:.1f}%)")

## 5. Baseline Model (Majority Class Classifier)

In [ ]:
# ── Majority class baseline ───────────────────────────────────────────────────
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

baseline_acc = accuracy_score(y_test, y_pred_dummy)
baseline_f1  = f1_score(y_test, y_pred_dummy, average='macro', zero_division=0)

print(f"Baseline Accuracy (majority class) : {baseline_acc:.4f}")
print(f"Baseline Macro F1                  : {baseline_f1:.4f}")

# Store all results for the final comparison table
results = []
results.append({'Model': 'Majority Class Baseline',
                'Accuracy': baseline_acc,
                'Macro F1': baseline_f1})

## 6. Traditional ML — Naïve Bayes

**Multinomial Naïve Bayes** is a probabilistic classifier well-suited for text classification.
It models word counts (via TF-IDF weights) as features, assumes conditional independence between
features given the class, and is computationally very efficient.  
Its main weakness is the independence assumption, which does not hold in natural language.

In [ ]:
# ── Naïve Bayes pipeline ──────────────────────────────────────────────────────
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                              sublinear_tf=True)),
    ('clf',   MultinomialNB())
])

# ── Hyperparameter tuning via GridSearchCV ────────────────────────────────────
nb_param_grid = {
    'tfidf__max_features': [10000, 20000],
    'tfidf__ngram_range':  [(1, 1), (1, 2)],
    'clf__alpha':          [0.1, 0.5, 1.0]
}

nb_cv = GridSearchCV(
    nb_pipeline,
    nb_param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
nb_cv.fit(X_train, y_train)

print("Best NB params :", nb_cv.best_params_)
print("Best CV F1     :", nb_cv.best_score_)

In [ ]:
# ── Evaluate best NB model on test set ───────────────────────────────────────
best_nb = nb_cv.best_estimator_
y_pred_nb = best_nb.predict(X_test)

nb_acc = accuracy_score(y_test, y_pred_nb)
nb_f1  = f1_score(y_test, y_pred_nb, average='macro')

print(f"Naïve Bayes — Test Accuracy : {nb_acc:.4f}")
print(f"Naïve Bayes — Macro F1      : {nb_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))

results.append({'Model': 'Naïve Bayes', 'Accuracy': nb_acc, 'Macro F1': nb_f1})

In [ ]:
# ── Confusion matrix — Naïve Bayes ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_nb,
    display_labels=le.classes_,
    ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45
)
ax.set_title('Confusion Matrix — Naïve Bayes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_cm_nb.png', dpi=150)
plt.show()

## 7. Traditional ML — Random Forest

**Random Forest** is an ensemble of decision trees trained on bootstrapped subsets of data and
random subsets of features. It handles high-dimensional TF-IDF spaces well, is robust to
overfitting, and provides feature-importance estimates.  
Its main drawbacks are higher memory use and slower inference compared to Naïve Bayes.

In [ ]:
# ── Random Forest pipeline ────────────────────────────────────────────────────
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                              sublinear_tf=True)),
    ('clf',   RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

# ── Hyperparameter tuning via GridSearchCV ────────────────────────────────────
rf_param_grid = {
    'tfidf__max_features': [20000],
    'clf__n_estimators':   [100, 200],
    'clf__max_depth':      [None, 30],
    'clf__min_samples_split': [2, 5]
}

rf_cv = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
rf_cv.fit(X_train, y_train)

print("Best RF params :", rf_cv.best_params_)
print("Best CV F1     :", rf_cv.best_score_)

In [ ]:
# ── Evaluate best RF model on test set ───────────────────────────────────────
best_rf = rf_cv.best_estimator_
y_pred_rf = best_rf.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf, average='macro')

print(f"Random Forest — Test Accuracy : {rf_acc:.4f}")
print(f"Random Forest — Macro F1      : {rf_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

results.append({'Model': 'Random Forest', 'Accuracy': rf_acc, 'Macro F1': rf_f1})

In [ ]:
# ── Confusion matrix — Random Forest ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=le.classes_,
    ax=ax, colorbar=False, cmap='Greens', xticks_rotation=45
)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_cm_rf.png', dpi=150)
plt.show()

In [ ]:
# ── Feature importance — top 20 TF-IDF terms ─────────────────────────────────
tfidf_vec = best_rf.named_steps['tfidf']
rf_clf    = best_rf.named_steps['clf']
feature_names = tfidf_vec.get_feature_names_out()
importances   = rf_clf.feature_importances_
top20_idx     = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(feature_names[top20_idx][::-1],
        importances[top20_idx][::-1],
        color=sns.color_palette('muted')[0])
ax.set_title('Top-20 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('fig_rf_importance.png', dpi=150)
plt.show()

## 8. Deep Learning Setup — Tokenisation & Sequence Padding

In [ ]:
# ── Sequence length analysis (used to justify MAX_LEN) ────────────────────────
# Tokenise all cleaned texts and measure word counts after cleaning
word_counts = df['clean_text'].str.split().str.len()

print("Word count statistics (after cleaning):")
print(word_counts.describe().round(1))
pcts = [50, 75, 90, 95, 99]
print("\nPercentiles:")
for p in pcts:
    print(f"  {p}th percentile : {int(np.percentile(word_counts, p))} words")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(word_counts, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(200, color='crimson', linestyle='--', lw=2, label='MAX_LEN = 200')
pct90 = int(np.percentile(word_counts, 90))
ax.axvline(pct90, color='orange', linestyle=':', lw=1.8,
           label=f'90th percentile = {pct90}')
ax.set_title('Distribution of Lyric Length (words, after cleaning)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Word Count', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig_seq_length.png', dpi=150)
plt.show()
print("Figure saved: fig_seq_length.png")
print(f"\nMAX_LEN=200 covers "
      f"{(word_counts <= 200).mean()*100:.1f}% of all lyrics.")


In [ ]:
# ── Keras tokeniser parameters ────────────────────────────────────────────────
VOCAB_SIZE  = 20000
MAX_LEN     = 200    # maximum sequence length (words)
EMBED_DIM   = 64     # embedding dimension

# Fit tokeniser on training texts only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

def texts_to_padded(texts, tokenizer, maxlen=MAX_LEN):
    """Convert text array to padded integer sequences."""
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')

X_train_seq = texts_to_padded(X_train, tokenizer)
X_val_seq   = texts_to_padded(X_val,   tokenizer)
X_test_seq  = texts_to_padded(X_test,  tokenizer)

# One-hot encode labels for Keras
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f"X_train_seq shape : {X_train_seq.shape}")
print(f"Vocabulary size   : {len(tokenizer.word_index):,}")

## 9. Deep Learning — Simple RNN

A **Simple RNN** processes sequences step-by-step, maintaining a hidden state that carries
information across time steps. It is a compact model that learns sequential patterns in lyrics.  
However, it suffers from the **vanishing gradient** problem on long sequences, making it less
effective than LSTM on lyrics with complex long-range dependencies.

In [ ]:
# ── Helper: plot training history ─────────────────────────────────────────────
def plot_history(history, model_name):
    """Plot training and validation accuracy and loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['accuracy'],     label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title(f'{model_name} — Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history.history['loss'],     label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title(f'{model_name} — Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    fname = f'fig_history_{model_name.lower().replace(" ", "_")}.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"Figure saved: {fname}")

In [ ]:
# ── Build Simple RNN ──────────────────────────────────────────────────────────
def build_rnn(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
              max_len=MAX_LEN, num_classes=NUM_CLASSES,
              units=64, dropout_rate=0.3, l2_reg=1e-4):
    """Construct a Simple RNN model for multi-class text classification."""
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        SimpleRNN(units, return_sequences=False,
                  kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        Dense(64, activation='relu', kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax')
    ], name='SimpleRNN')
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

rnn_model = build_rnn()
rnn_model.summary()

In [ ]:
# ── Train Simple RNN ──────────────────────────────────────────────────────────
callbacks_rnn = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

rnn_history = rnn_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=20,
    batch_size=64,
    callbacks=callbacks_rnn,
    verbose=1
)

In [ ]:
# ── Plot RNN training history ─────────────────────────────────────────────────
plot_history(rnn_history, 'Simple RNN')

In [ ]:
# ── Evaluate Simple RNN ───────────────────────────────────────────────────────
y_pred_rnn_prob = rnn_model.predict(X_test_seq)
y_pred_rnn = np.argmax(y_pred_rnn_prob, axis=1)

rnn_acc = accuracy_score(y_test, y_pred_rnn)
rnn_f1  = f1_score(y_test, y_pred_rnn, average='macro')

print(f"Simple RNN — Test Accuracy : {rnn_acc:.4f}")
print(f"Simple RNN — Macro F1      : {rnn_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rnn, target_names=le.classes_))

results.append({'Model': 'Simple RNN', 'Accuracy': rnn_acc, 'Macro F1': rnn_f1})

In [ ]:
# ── Confusion matrix — Simple RNN ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rnn,
    display_labels=le.classes_,
    ax=ax, colorbar=False, cmap='Oranges', xticks_rotation=45
)
ax.set_title('Confusion Matrix — Simple RNN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_cm_rnn.png', dpi=150)
plt.show()

## 10. Deep Learning — LSTM

**Long Short-Term Memory (LSTM)** extends Simple RNN with three gating mechanisms
(input, forget, output) that allow the model to selectively retain or discard information
across long sequences, effectively addressing the vanishing gradient problem.  
This makes LSTM well-suited for longer song lyrics with complex thematic structure.
Its drawback is higher computational cost compared to SimpleRNN.

In [ ]:
# ── Build LSTM ────────────────────────────────────────────────────────────────
def build_lstm(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
               max_len=MAX_LEN, num_classes=NUM_CLASSES,
               units=128, dropout_rate=0.4, l2_reg=1e-4):
    """Construct a Bidirectional LSTM model for multi-class text classification."""
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        Bidirectional(LSTM(units, return_sequences=True,
                           kernel_regularizer=l2(l2_reg))),
        Dropout(dropout_rate),
        Bidirectional(LSTM(units // 2,
                           kernel_regularizer=l2(l2_reg))),
        Dropout(dropout_rate),
        Dense(64, activation='relu', kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax')
    ], name='BiLSTM')
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm()
lstm_model.summary()

In [ ]:
# ── Train LSTM ────────────────────────────────────────────────────────────────
callbacks_lstm = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

lstm_history = lstm_model.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=25,
    batch_size=64,
    callbacks=callbacks_lstm,
    verbose=1
)

In [ ]:
# ── Plot LSTM training history ────────────────────────────────────────────────
plot_history(lstm_history, 'LSTM')

In [ ]:
# ── Evaluate LSTM ─────────────────────────────────────────────────────────────
y_pred_lstm_prob = lstm_model.predict(X_test_seq)
y_pred_lstm = np.argmax(y_pred_lstm_prob, axis=1)

lstm_acc = accuracy_score(y_test, y_pred_lstm)
lstm_f1  = f1_score(y_test, y_pred_lstm, average='macro')

print(f"LSTM — Test Accuracy : {lstm_acc:.4f}")
print(f"LSTM — Macro F1      : {lstm_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lstm, target_names=le.classes_))

results.append({'Model': 'LSTM (Bidirectional)', 'Accuracy': lstm_acc, 'Macro F1': lstm_f1})

In [ ]:
# ── Confusion matrix — LSTM ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lstm,
    display_labels=le.classes_,
    ax=ax, colorbar=False, cmap='Purples', xticks_rotation=45
)
ax.set_title('Confusion Matrix — LSTM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_cm_lstm.png', dpi=150)
plt.show()

## 11. Evaluation — ROC-AUC Curves (One-vs-Rest)

In [ ]:
# ── Multi-class ROC-AUC helper ────────────────────────────────────────────────
def plot_roc_multiclass(y_true, y_score, class_names, title, filename):
    """
    Plot One-vs-Rest ROC curves for each class and compute macro-average AUC.
    y_score: probability matrix of shape (n_samples, n_classes)
    """
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    fpr_grid = np.linspace(0, 1, 200)
    tprs = []
    aucs = []

    fig, ax = plt.subplots(figsize=(9, 6))
    palette = sns.color_palette('tab10', len(class_names))

    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        tprs.append(np.interp(fpr_grid, fpr, tpr))
        ax.plot(fpr, tpr, lw=1.2, color=palette[i],
                label=f'{cls} (AUC={roc_auc:.2f})')

    mean_tpr = np.mean(tprs, axis=0)
    macro_auc = auc(fpr_grid, mean_tpr)
    ax.plot(fpr_grid, mean_tpr, color='black', lw=2.5, linestyle='--',
            label=f'Macro avg (AUC={macro_auc:.2f})')
    ax.plot([0, 1], [0, 1], 'k:', lw=0.8)

    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"Figure saved: {filename}")
    return macro_auc

In [ ]:
# ── ROC — Naïve Bayes ─────────────────────────────────────────────────────────
y_score_nb = best_nb.predict_proba(X_test)
roc_nb = plot_roc_multiclass(
    y_test, y_score_nb, le.classes_,
    'ROC Curves — Naïve Bayes', 'fig_roc_nb.png'
)

In [ ]:
# ── ROC — Random Forest ───────────────────────────────────────────────────────
y_score_rf = best_rf.predict_proba(X_test)
roc_rf = plot_roc_multiclass(
    y_test, y_score_rf, le.classes_,
    'ROC Curves — Random Forest', 'fig_roc_rf.png'
)

In [ ]:
# ── ROC — Simple RNN ──────────────────────────────────────────────────────────
roc_rnn = plot_roc_multiclass(
    y_test, y_pred_rnn_prob, le.classes_,
    'ROC Curves — Simple RNN', 'fig_roc_rnn.png'
)

In [ ]:
# ── ROC — LSTM ────────────────────────────────────────────────────────────────
roc_lstm = plot_roc_multiclass(
    y_test, y_pred_lstm_prob, le.classes_,
    'ROC Curves — LSTM', 'fig_roc_lstm.png'
)

## 12. Final Results Table & Discussion

In [ ]:
# ── Append ROC-AUC to results ─────────────────────────────────────────────────
roc_map = {
    'Naïve Bayes':           roc_nb,
    'Random Forest':         roc_rf,
    'Simple RNN':            roc_rnn,
    'LSTM (Bidirectional)':  roc_lstm
}

results_df = pd.DataFrame(results)
results_df['Macro ROC-AUC'] = results_df['Model'].map(
    lambda m: roc_map.get(m, np.nan)
)
results_df = results_df.round(4)

print("\n=== Final Results ===")
print(results_df.to_string(index=False))

In [ ]:
# ── Visualisation: grouped bar chart of all metrics ───────────────────────────
models_with_metrics = results_df[results_df['Model'] != 'Majority Class Baseline']

x  = np.arange(len(models_with_metrics))
w  = 0.25
metrics = ['Accuracy', 'Macro F1', 'Macro ROC-AUC']
colors  = sns.color_palette('Set1', 3)

fig, ax = plt.subplots(figsize=(11, 5))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i * w, models_with_metrics[metric].values, w,
                  label=metric, color=color, alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)

ax.set_xticks(x + w)
ax.set_xticklabels(models_with_metrics['Model'].values, fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.axhline(y=baseline_acc, color='grey', linestyle='--', lw=1.2,
           label=f'Baseline Acc = {baseline_acc:.3f}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_model_comparison.png', dpi=150)
plt.show()
print("Figure saved: fig_model_comparison.png")

In [ ]:
# ── Styled HTML table for the report ─────────────────────────────────────────
print(results_df.to_markdown(index=False))

## 13. Summary

This notebook implemented a complete NLP pipeline for classifying song lyrics into eight topic
categories using the Music Dataset (1950–2019):

| Step | Details |
|---|---|
| Preprocessing | Lowercase, regex cleaning, stopword removal, Porter stemming |
| Baseline | Majority class classifier |
| Traditional ML | Naïve Bayes, Random Forest (with GridSearchCV hyperparameter tuning) |
| Deep Learning | Simple RNN, Bidirectional LSTM (with EarlyStopping, ReduceLROnPlateau) |
| Metrics | Accuracy, Macro F1, Macro ROC-AUC, classification report, confusion matrices |

The Bidirectional LSTM is expected to outperform Simple RNN on longer lyric sequences due to its
gating mechanisms, while Random Forest typically outperforms Naïve Bayes thanks to its ensemble
approach and ability to capture feature interactions.

All figures are saved to disk alongside this notebook for inclusion in the written report.